# Unittest of TRT Engine using Sionna 2.0 (PyTorch)

In [1]:
import torch
import numpy as np

from sionna.phy.mapping import Demapper
from sionna.phy.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver
from sionna.phy.channel import AWGN
from sionna.phy.utils import hard_decisions


In [2]:
channel = AWGN()

def generate_test_data(num_ofdm_symbols, num_prbs, num_rx_ant,
                       num_add_pilot_positions, scaling_factor):

    # Set the PUSCH configuration parameters
    pusch_config = PUSCHConfig()
    pusch_config.mapping_type = "B" # allows DMRS at pos 0,5,10
    pusch_config.dmrs.num_cdm_groups_without_data = 1 # no multi-user
    pusch_config.dmrs.dmrs_port_set = [0] # we use only first port (no multi-user)
    pusch_config.dmrs.additional_position = num_add_pilot_positions
    pusch_config.symbol_allocation = [0,num_ofdm_symbols]
    pusch_config.n_size_bwp = num_prbs
    pusch_config.n_rnti = 1
    pusch_config.carrier.subcarrier_spacing = 30
    pusch_config.carrier.slot_number = 0
    pusch_config.dmrs.n_id = 0
    pusch_config.dmrs.n_scid = 0
    pusch_config.tb.n_id = 0

    # Instantiate a PUSCHTransmitter from the PUSCHConfig
    pusch_transmitter = PUSCHTransmitter(pusch_config)
    pusch_receiver = PUSCHReceiver(pusch_transmitter, return_tb_crc_status=True)

    pilot_pos = torch.nonzero(pusch_transmitter.pilot_pattern.mask[0,0,:]==1).detach().cpu().numpy()
    # dmrs ofdm positions
    dmrs_ofdm_pos = np.unique(pilot_pos[:,0])
    dmrs_ofdm_pos = np.sort(dmrs_ofdm_pos)

    prb_pilot_pos = np.unique(pilot_pos[:,1])
    prb_pilot_pos = prb_pilot_pos[prb_pilot_pos < 12]
    prb_pilot_pos = np.sort(prb_pilot_pos)

    x, bits = pusch_transmitter(1)

    slot = x.squeeze(0).squeeze(0) # remove unused dimensions

    # ground truth bits in resourcegrid
    demapper = Demapper("maxlog","qam", 4)
    bits_rg = hard_decisions(demapper(slot[..., None], 0.01))
    bits_rg = bits_rg.squeeze(0)

    # duplicated antennas (for testing only)
    slot = torch.tile(slot, (num_rx_ant, 1, 1))

    # add noise
    no = 0.1
    y = channel(slot, no)
    # y has shape [num_rx_ant, num_ofdm_symbols, num_subcarriers]

    # generate noisy channel estimates
    p_idx = torch.nonzero(pusch_transmitter.pilot_pattern.mask[0,:]==1)
    ch_est = slot[tuple(p_idx.t())]
    no_chest = 0.1
    ch_est_n = channel(ch_est, no_chest)

    # draw random complex phase rotation (on same device as y)
    device = y.device
    phase = torch.rand(1, device=device) * 2 * np.pi
    phase = torch.exp(torch.complex(torch.zeros(1, device=device), phase))

    # apply scaling
    y = y * scaling_factor * phase
    ch_est_n = ch_est_n * scaling_factor * phase

    return y, ch_est_n, bits_rg, dmrs_ofdm_pos, prb_pilot_pos


In [3]:
num_ofdm_symbols = [3, 5, 13] # between 1 and 14
num_prbs = [1, 5, 106] # between 1 and 273
num_rx_ant = [1, 2, 4]
scaling_factor = [5.0, 10.0, 20.0] # scale outputs
num_add_pilot_positions = [0, 1, 2, 3]

for num_ofdm_symbols_ in num_ofdm_symbols:
    for num_prbs_ in num_prbs:
        for num_rx_ant_ in num_rx_ant:
            for scaling_factor_ in scaling_factor:
                for num_add_pilot_positions_ in num_add_pilot_positions:

                    y, ch_est_n, bits_rg, dmrs_ofdm_pos, prb_pilot_pos = generate_test_data(num_ofdm_symbols_, num_prbs_, num_rx_ant_, num_add_pilot_positions_, scaling_factor_)

                    print("--------------------------------")
                    print("y:", y.shape)
                    print("ch_est:", ch_est_n.shape)
                    print("bits_rg:", bits_rg.shape)
                    print("dmrs_ofdm_pos:", dmrs_ofdm_pos)
                    print("prb_pilot_pos:", prb_pilot_pos)


--------------------------------
y: torch.Size([1, 3, 12])
ch_est: torch.Size([6])
bits_rg: torch.Size([3, 12, 4])
dmrs_ofdm_pos: [0]
prb_pilot_pos: [ 0  2  4  6  8 10]
--------------------------------
y: torch.Size([1, 3, 12])
ch_est: torch.Size([6])
bits_rg: torch.Size([3, 12, 4])
dmrs_ofdm_pos: [0]
prb_pilot_pos: [ 0  2  4  6  8 10]
--------------------------------
y: torch.Size([1, 3, 12])
ch_est: torch.Size([6])
bits_rg: torch.Size([3, 12, 4])
dmrs_ofdm_pos: [0]
prb_pilot_pos: [ 0  2  4  6  8 10]
--------------------------------
y: torch.Size([1, 3, 12])
ch_est: torch.Size([6])
bits_rg: torch.Size([3, 12, 4])
dmrs_ofdm_pos: [0]
prb_pilot_pos: [ 0  2  4  6  8 10]
--------------------------------
y: torch.Size([1, 3, 12])
ch_est: torch.Size([6])
bits_rg: torch.Size([3, 12, 4])
dmrs_ofdm_pos: [0]
prb_pilot_pos: [ 0  2  4  6  8 10]
--------------------------------
y: torch.Size([1, 3, 12])
ch_est: torch.Size([6])
bits_rg: torch.Size([3, 12, 4])
dmrs_ofdm_pos: [0]
prb_pilot_pos: [ 0  2